# Faruq-v3 — label identifiability audit

Audit CPU-only ini menguji apakah label ukuran kecil/sedang/besar memiliki sinyal geometri yang dapat diamati oleh model. Tidak ada training dan test tidak tersedia.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
import coffee_detector
os.chdir(REPO)
print('IMPORT:', coffee_detector.__file__)

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_diagnostic.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DIAGNOSTIC = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_diagnostic.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_label_identifiability.json'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT   :', PROJECT_ROOT)
print('DATA      :', DATA_ROOT)
print('DIAGNOSTIC:', DIAGNOSTIC)
print('OUTPUT    :', OUTPUT)

In [ ]:
from coffee_detector.analysis.faruq_v3_label_identifiability import audit_faruq_v3_label_identifiability

result = audit_faruq_v3_label_identifiability(
    DATA_ROOT, OUTPUT, diagnostic=DIAGNOSTIC
)
print('AUDIT SELESAI:', result['decision'])

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for split, split_report in result['split_reports'].items():
    for family, family_report in split_report['size_families'].items():
        feature = family_report['best_feature']
        medians = family_report['features'][feature]['medians'] if feature else {}
        rows.append({
            'split': split,
            'family': family,
            'n_small': family_report['samples']['small'],
            'n_medium': family_report['samples']['medium'],
            'n_large': family_report['samples']['large'],
            'best_feature': feature,
            'macro_order_auc': family_report['best_macro_order_auc'],
            'median_small': medians.get('small'),
            'median_medium': medians.get('medium'),
            'median_large': medians.get('large'),
            'geometry_signal': family_report['geometry_signal'],
        })
family_table = pd.DataFrame(rows)
confusion_table = pd.DataFrame(result['confusion_taxonomy']['rows'])
display(family_table.style.format({'macro_order_auc': '{:.3f}', 'median_small': '{:.5f}', 'median_medium': '{:.5f}', 'median_large': '{:.5f}'}))
display(confusion_table)
print('CONFUSION COUNTS:', result['confusion_taxonomy']['counts_by_kind'])
print('DECISION:', result['decision'])
print('NEXT    :', result['next_action'])
print('TRAINING:', result['training_executed'])
print('TEST    :', result['test_images_accessed'])
print('SUMMARY :', OUTPUT)
print('Kirim tabel ini. Jangan training model baru sebelum keputusan audit dinilai.')